In [1]:
%load_ext autoreload
%autoreload 2
from logsProcessing_export import display_round_stripe, round_list_tpl, Log

Loading logs from logs...
2504 files found.
DONE. Loaded 2504 completed game logs.


In [3]:
# device = "cuda:1"

In [4]:
import torch

In [5]:
import re
from PIL import Image
from llava.model.builder import load_pretrained_model
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from matplotlib import pyplot as plt

/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model_path = "liuhaotian/llava-v1.5-7b"
tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, None, model_name="llava_v1_5", 
                                                                       device_map="auto", 
                                                                       # device_map=device, 
                                                                       torch_dtype=torch.float16)

/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards:   0%|                                                                                                                               | 0/2 [00:00<?, ?it/s]/srv/data/gusloryst/llava3.10/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.stora

In [ ]:
round_list_tpl[7]

In [ ]:
seven = display_round_stripe(round_list_tpl[0], show=True)

In [ ]:
seven

In [7]:
def run_turn(turn):
    joined_list = turn[0].images['A'] + turn[0].images['B']
    joined_list_raw = [(re.search(r'_0*(\d+)\.jpg$', path)).group(1) for path in joined_list]
    
    missing_path = "images/"
    joined_path_list = [missing_path + a for a in joined_list]
    images_tensor = process_images(
            [Image.open(image) for image in joined_path_list],
            image_processor,
            model.config
        ).to(model.device, dtype=torch.float16)
    img = images_tensor
    
    text_history = []

    for message in turn[0].messages:
        if message.type == "text":
            text_history.append("{}: {}".format(message.speaker, message.text))
        if message.type == "selection":
            label = "common" if message.text.split()[1] == "<com>" else "different"
            text_history.append("{} marks image {} as {}".format(message.speaker, Log.strip_image_id(message.text.split()[2]), label))

    text_history_missing = text_history[:-1]
    text_history_last = text_history[-1:]

    prompt_for_generation = f"""
                            You are a helpful language and vision assistant. You see a chat between two people, A and B. They are playing a game in which they are seeing set of 6 images from the photobook album. Their task is to find out which photos are common for both of them, and which are different. They are chatting between eachother. First 6 images you received are A's view, and the other 6 are B's view. Each player does not see what the other player sees.
                            
                            What do you think is the next message based on the information you have about the game, the players, the images they see, and their chat?
                            
                            CHAT:
                            
                            {text_history_missing}
                            """

    
    full_input_ids = tokenizer_image_token(prompt_for_generation, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
    full_input_ids = full_input_ids.unsqueeze(0).to(model.device)
    #get the length of the target message
    targets_as_input_ids = tokenizer_image_token(text_history_last[0], tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
    target_len = targets_as_input_ids.unsqueeze(0).to(model.device).shape[1]
    # calculate loss, i.e. for perplexity calculation
    # https://huggingface.co/docs/transformers/v4.37.2/en/perplexity
    # https://huggingface.co/spaces/evaluate-metric/perplexity
    targets_as_input_ids = tokenizer_image_token(text_history_last[0], tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
    targets_as_input_ids = targets_as_input_ids.unsqueeze(0).to(model.device)

    initial_context_len = full_input_ids.shape[-1] - target_len
    seq_len = target_len
    nlls = []
    begin_loc = 0
    prev_end_loc = initial_context_len
    stride = 1
    for end_loc in range(initial_context_len, initial_context_len + seq_len, stride):
        input_ids = full_input_ids[:, begin_loc:end_loc].to(model.device)
        target_ids = input_ids.clone()
        target_ids[:, :-1] = -100
        with torch.no_grad():
            outputs = model(input_ids, images=img, image_sizes=[(336, 336)], labels=target_ids)
            loss = outputs.loss
            neg_log_likelihood = loss
        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        stride += 1
        if end_loc == initial_context_len:
            break
    ppl = torch.exp(torch.stack(nlls).mean())
    
    
    # print(text_history)
    # print(text_history_missing)
    # print(text_history_last)
    # print(images_tensor)

In [8]:
run_turn(round_list_tpl[7])